In [3]:
import os
from google.colab import userdata
# Retrieve token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# Authenticate with Hugging Face Hub
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
import torch
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "/content/train.csv",
                                          "test": "/content/test.csv"})

print("Loaded csv files with load_dataset.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded csv files with load_dataset.


In [5]:
def combine_columns(example):
    example['combined_text'] = f"{example['prompt']} {example['A']}"
    return example

dataset = dataset.map(combine_columns)

print(f"Character length of the combined text is: {len(dataset['train'][51]['combined_text'])}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Character length of the combined text is: 614


In [6]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_pretrained('bert-base-uncased')

print(f"Total Vocabulary size: {tokenizer.get_vocab_size()}")

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Total Vocabulary size: 30522


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print(f"Total Vocabulary size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Total Vocabulary size: 30522


In [8]:
# print(f"Integer ID for the [SEP] token: {tokenizer.get_vocab()['[SEP]']}")
print(f"Integer ID for the [SEP] token: {tokenizer.vocab['[SEP]']}")

Integer ID for the [SEP] token: 102


In [9]:
def bert_tokenize(batch):
    # Extract the text column to tokenize
    return tokenizer(batch['prompt'],
                     padding='max_length',
                     truncation=True,
                     max_length=128)

In [10]:
tokenized_dataset = dataset['train'].map(bert_tokenize, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [11]:
import torch
import sys
from unittest.mock import patch

# Set format to torch to handle tensors
# We use a context manager to temporarily hide torchvision to bypass a known ImportError in the datasets library
with patch.dict(sys.modules, {'torchvision': None}):
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask'])

# To get the shape, we must slice the column to return the actual tensor
input_ids_tensor = tokenized_dataset['input_ids']
input_ids_shape = input_ids_tensor[:].shape

print(f"The dimensions of the input_ids tensor are: {input_ids_shape}")

The dimensions of the input_ids tensor are: torch.Size([2000, 128])


In [12]:
from transformers import AutoModelForSequenceClassification

model_name = "bert-base-uncased"

bert_model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
hidden_size = bert_model.config.hidden_size
num_heads = bert_model.config.num_attention_heads
dim_per_head = hidden_size // num_heads

print(f"Hidden Size: {hidden_size}")
print(f"Number of Attention Heads: {num_heads}")
print(f"Dimensionality per Attention Head: {dim_per_head}")

Hidden Size: 768
Number of Attention Heads: 12
Dimensionality per Attention Head: 64


In [14]:
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"

bert_model = AutoModel.from_pretrained(model_name)
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
prompt = dataset['train']['prompt'][0]
tokenized_text = bert_tokenizer(prompt, return_tensors='pt')
num_tokens = tokenized_text['input_ids'].shape[1]
print(f"Number of tokens (Sequence Length): {num_tokens}")

with torch.no_grad():
    outputs = bert_model(**tokenized_text)

print(f"The exact shape of last_hidden_state is: {outputs.last_hidden_state.shape}")

Number of tokens (Sequence Length): 31
The exact shape of last_hidden_state is: torch.Size([1, 31, 768])


In [16]:
first_five = outputs.last_hidden_state[0, 0, :5]

cls_sum = torch.sum(first_five).item()
print(f"First 5 values: {first_five}")
print(f"Sum of first 5 values: {round(cls_sum, 4)}")

First 5 values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Sum of first 5 values: -1.2001


In [17]:
from transformers import AutoModel, AutoTokenizer

bert_customized_model = AutoModel.from_pretrained(model_name, output_attentions=True)
bert_customized_tokenizer = AutoTokenizer.from_pretrained(model_name, output_attentions=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
qs_prompt = "Light-ion fusion is a technique."

inputs = bert_customized_tokenizer(qs_prompt, return_tensors='pt')

with torch.no_grad():
    outputs = bert_customized_model(**inputs)

# Extracting: [Last Layer][-1], [Batch 0][0], [Head 0][0]
last_layer_first_head_attention = outputs.attentions[-1][0][0]

print(f"Shape of the specific attention matrix: {last_layer_first_head_attention.shape}")
print(last_layer_first_head_attention)

Shape of the specific attention matrix: torch.Size([10, 10])
tensor([[0.1148, 0.0409, 0.0698, 0.0579, 0.1025, 0.0245, 0.0211, 0.1091, 0.3029,
         0.1566],
        [0.0054, 0.0203, 0.0261, 0.0314, 0.0172, 0.0015, 0.0016, 0.0062, 0.7036,
         0.1868],
        [0.0030, 0.0169, 0.0210, 0.0260, 0.0210, 0.0026, 0.0026, 0.0060, 0.7046,
         0.1963],
        [0.0038, 0.0074, 0.0094, 0.0102, 0.0080, 0.0014, 0.0011, 0.0021, 0.7684,
         0.1882],
        [0.0076, 0.0101, 0.0214, 0.0278, 0.0436, 0.0030, 0.0032, 0.0079, 0.6978,
         0.1774],
        [0.0208, 0.0280, 0.0470, 0.0395, 0.0672, 0.0144, 0.0119, 0.0348, 0.5359,
         0.2005],
        [0.0118, 0.0293, 0.0409, 0.0384, 0.0636, 0.0111, 0.0089, 0.0310, 0.5714,
         0.1937],
        [0.0138, 0.0233, 0.0498, 0.0391, 0.1433, 0.0079, 0.0076, 0.0322, 0.5015,
         0.1815],
        [0.0042, 0.0038, 0.0023, 0.0025, 0.0041, 0.0028, 0.0018, 0.0027, 0.7664,
         0.2094],
        [0.0033, 0.0034, 0.0021, 0.0027, 0.0040,

In [19]:
tokens = bert_customized_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
fusion_index = tokens.index('fusion')

attention_weight = last_layer_first_head_attention[0, fusion_index].item()

print(f"Tokens: {tokens}")
print(f"Index of 'fusion': {fusion_index}")
print(f"Attention weight from [CLS] to 'fusion': {round(attention_weight, 4)}")

Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion': 4
Attention weight from [CLS] to 'fusion': 0.1025


In [20]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

st_prompt = dataset['train']['prompt'][0]
st_optB = dataset['train']['B'][0]

st_prompt_embedding = model.encode(st_prompt)
st_optB_embedding = model.encode(st_optB)

similarity = cos_sim(st_prompt_embedding, st_optB_embedding)
print(f"Similarity Score: {float(similarity):.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity Score: 0.7658


In [21]:
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'

sentence_transformers_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

def get_top_3_predictions(sample):
    prompt = sample['prompt']
    options = [sample[key] for key in ['A', 'B', 'C', 'D', 'E']]

    prompt_embedding = sentence_transformers_model.encode(prompt, convert_to_tensor=True)
    option_embeddings = sentence_transformers_model.encode(options, convert_to_tensor=True)

    cosine_scores = cos_sim(prompt_embedding, option_embeddings)[0]

    top_results = torch.topk(cosine_scores, k=3)

    labels = ['A', 'B', 'C', 'D', 'E']
    predictions = [labels[idx] for idx in top_results.indices.tolist()]

    return {"top_3_predictions": " ".join(predictions)}

print("Processing the entire training dataset... ")
results = dataset['train'].map(get_top_3_predictions)

display(pd.DataFrame(results).head())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processing the entire training dataset... 


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

,id,prompt,A,B,C,D,E,answer,combined_text,top_3_predictions
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,Pick the best possible answer: What is Martin ...,C D B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,What is accelerator-based light-ion fusion? Ac...,B C D
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,Determine the correct option: What is the term...,B A D
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,Select the most accurate option: What is Marti...,C D B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,Identify the correct statement: What is the co...,E D A
